[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/04_decoding/04_decoding_strategies.ipynb)

# 04 · 解码策略全手写 — Decoding Strategies from Scratch

<span style="background:#1a7f37;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 PyTorch，无外部模型/数据下载，全程 CPU 可跑（mini-GPT 训练仅需几秒）。

配套讲解：`04_讲解.html`。本 notebook 的主线：

1. 在一个**手工设计的教学 logits 向量**上，亲手实现 `sample_greedy / sample_temperature / sample_top_k / sample_top_p`（全部直接操作 logits）；
2. 可视化 temperature 与 top-p 截断的几何效果；
3. 内嵌一个精简 mini-GPT（复用模块 03 的结构，快速训 ~300 步），亲眼看 **greedy 重复退化** vs top-p 不退化，以及 repetition penalty 的效果与副作用；
4. 用两个固定 categorical 分布模拟 **speculative decoding** [Leviathan 2022]，蒙特卡洛验证"输出分布 = target 分布 p"与接受率公式；
5. 3 道 ✏️ 练习 + 📖 参考答案。

引用：top-p [Holtzman 2019, arXiv:1904.09751]、top-k [Fan 2018, arXiv:1805.04833]、speculative [Leviathan 2022, arXiv:2211.17192]。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)

# ---- 教学用固定 logits：两种典型形态（词表大小 8，token 记为 A..H）----
TOKENS = list("ABCDEFGH")

# 尖峰形态：下一个 token 几乎确定（如 "United" 后面接 "States"）
logits_peaked = torch.tensor([5.0, 2.0, 1.0, 0.5, 0.0, -1.0, -2.0, -3.0])
# 平坦形态：许多延续都合理（如开放叙事的句首）
logits_flat   = torch.tensor([1.2, 1.1, 1.0, 0.95, 0.9, 0.85, 0.5, 0.3])

def show_dist(name, logits):
    p = F.softmax(logits, dim=-1)
    bars = "  ".join(f"{t}:{pi:.3f}" for t, pi in zip(TOKENS, p))
    print(f"{name:8s} {bars}")

show_dist("peaked", logits_peaked)
show_dist("flat",   logits_flat)
# 注意：模型每一步前向输出的就是这样一个 logits 向量。
# 本 notebook 的所有解码策略都只做一件事：决定如何把它变成一个具体 token。

In [ ]:
# ---- 四个解码原语：全部在 logits 上操作，返回采样到的 token id（int）----

def sample_greedy(logits):
    # T -> 0 的极限：直接取 argmax
    return int(torch.argmax(logits).item())

def sample_temperature(logits, T=1.0):
    # p_i = softmax(z_i / T)；T<1 分布更尖，T>1 更平（排序永远不变）
    probs = F.softmax(logits / T, dim=-1)
    return int(torch.multinomial(probs, 1).item())

def sample_top_k(logits, k, T=1.0):
    # 只保留 logit 最大的 k 个，其余置 -inf 后再 softmax 采样 [Fan 2018]
    v, _ = torch.topk(logits, k)
    cutoff = v[-1]
    masked = torch.where(logits >= cutoff, logits, torch.tensor(float("-inf")))
    return sample_temperature(masked, T)

def sample_top_p(logits, p, T=1.0):
    # nucleus sampling [Holtzman 2019]：按概率降序累积，
    # 保留"累计概率首次达到 >= p 的最小集合"（跨过阈值的那个 token 必须保留）
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits / T, dim=-1)
    cum = torch.cumsum(probs, dim=-1)
    excl = cum - probs            # 该 token 之前已累计的质量（exclusive cumsum）
    keep = excl < p               # excl[0]=0 < p：概率最高的 token 永远保留
    masked = torch.where(keep, sorted_logits, torch.tensor(float("-inf")))
    j = sample_temperature(masked, T)     # 在排序后的空间里采样
    return int(sorted_idx[j].item())      # 映射回原 token id

# ---- 正确性体检 1：T=1 纯采样的频率应收敛到 softmax 概率 ----
N = 5000
counts = torch.zeros(8)
for _ in range(N):
    counts[sample_temperature(logits_peaked, T=1.0)] += 1
emp = counts / N
ref = F.softmax(logits_peaked, dim=-1)
print("empirical:", [f"{x:.3f}" for x in emp.tolist()])
print("softmax  :", [f"{x:.3f}" for x in ref.tolist()])
assert (emp - ref).abs().max() < 0.03, "采样频率应接近 softmax 概率"

# ---- 正确性体检 2：top-p 的自适应性（同一 p，两种形态下保留集大小悬殊）----
def top_p_kept_size(logits, p):
    probs = F.softmax(logits, dim=-1)
    sp, _ = torch.sort(probs, descending=True)
    excl = torch.cumsum(sp, dim=-1) - sp
    return int((excl < p).sum().item())

for name, lg in [("peaked", logits_peaked), ("flat", logits_flat)]:
    print(f"top-p(0.9) on {name:6s}: 保留 {top_p_kept_size(lg, 0.9)}/8 个 token   "
          f"(固定 top-k 无法同时适配两种形态)")
print("✅ 四个采样原语就绪")

In [ ]:
# ---- 可视化 1：同一 logits 在 T=0.5 / 1.0 / 2.0 下的分布 ----
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, T in zip(axes, [0.5, 1.0, 2.0]):
    p = F.softmax(logits_peaked / T, dim=-1)
    ax.bar(TOKENS, p.tolist(), color="#4878cf")
    ax.set_title(f"T = {T}   (entropy = {-(p * p.log()).sum():.2f} nats)")
    ax.set_ylim(0, 1)
axes[0].set_ylabel("prob")
fig.suptitle("temperature：T 小 → 尖（低熵）；T 大 → 平（高熵）；排序不变", y=1.05)
plt.tight_layout(); plt.show()

# ---- 可视化 2：top-p 截断边界（flat 分布, p=0.9）----
probs = F.softmax(logits_flat, dim=-1)
sp, si = torch.sort(probs, descending=True)
cum = torch.cumsum(sp, dim=-1)
kept = top_p_kept_size(logits_flat, 0.9)

fig, ax = plt.subplots(figsize=(7, 3.2))
colors = ["#4878cf" if i < kept else "#c44e52" for i in range(8)]
ax.bar(range(8), sp.tolist(), color=colors,
       tick_label=[TOKENS[i] for i in si.tolist()])
ax2 = ax.twinx()
ax2.plot(range(8), cum.tolist(), "k.-", label="cumulative prob")
ax2.axhline(0.9, ls="--", c="gray"); ax2.set_ylim(0, 1.05)
ax.axvline(kept - 0.5, ls=":", c="#c44e52")
ax.set_title(f"top-p=0.9 on flat：保留前 {kept} 个（蓝），其余清零（红）")
ax.set_ylabel("prob"); ax2.set_ylabel("cumsum")
plt.tight_layout(); plt.show()

## 真实模型上的解码：mini-GPT 重复退化实验

固定分布讲清了"每一步"的机制；但 **likelihood trap / 重复退化** [Holtzman 2019] 是序列级现象——错误会通过上下文自我强化。下面内嵌一个精简版模块 03 mini-GPT（字符级，2 层，约 0.1M 参数），在一小段文本上快训 ~300 步（CPU 仅需几秒）。**故意欠训练**：这正是退化现象最容易观察的状态。

In [ ]:
# ---- 精简 mini-GPT（与模块 03 同构：embedding + 2 层 pre-LN block + lm head）----
CORPUS = (
    "language models predict the next token. decoding turns those "
    "predictions into text. greedy decoding picks the most likely token "
    "at every step. sampling draws tokens at random from the distribution. "
    "temperature controls how sharp the distribution is. top k keeps only "
    "the k most likely tokens. top p keeps the smallest set of tokens whose "
    "total probability reaches p. repetition is the curse of greedy "
    "decoding. good text balances likelihood and diversity. "
) * 3
chars = sorted(set(CORPUS))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V, BLOCK = len(chars), 64
data = torch.tensor([stoi[c] for c in CORPUS])
print(f"corpus: {len(CORPUS)} chars, vocab = {V}")

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        T_ = x.shape[1]
        mask = torch.triu(torch.ones(T_, T_, dtype=torch.bool), diagonal=1)  # 因果掩码
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + a
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self, V, d=64, h=4, L=2):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(BLOCK, d)
        self.blocks = nn.Sequential(*[Block(d, h) for _ in range(L)])
        self.ln_f, self.head = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(self, idx):
        T_ = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T_))
        return self.head(self.ln_f(self.blocks(x)))   # (B, T, V) logits

model = MiniGPT(V)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):                       # 快速训练 ~300 步
    ix = torch.randint(len(data) - BLOCK - 1, (32,))
    xb = torch.stack([data[i:i + BLOCK] for i in ix])
    yb = torch.stack([data[i + 1:i + BLOCK + 1] for i in ix])
    loss = F.cross_entropy(model(xb).reshape(-1, V), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0 or step == 299:
        print(f"step {step:3d}  loss {loss.item():.3f}")
model.eval();

In [ ]:
# ---- 重复退化演示：greedy vs top-p（同一模型！）----
@torch.no_grad()
def generate(prompt, n, step_fn):
    # step_fn: logits(向量) -> token id；解码策略被干净地隔离在这一个回调里
    ids = [stoi[c] for c in prompt]
    for _ in range(n):
        ctx = torch.tensor(ids[-BLOCK:]).unsqueeze(0)
        logits = model(ctx)[0, -1]            # 只看最后一个位置的 logits
        ids.append(step_fn(logits))
    return "".join(itos[i] for i in ids)

def longest_loop(text, n=20):
    # 检测 n-gram 是否出现 >= 3 次（重复退化的信号）
    best = 0
    for i in range(len(text) - n):
        best = max(best, text.count(text[i:i + n]))
    return best

torch.manual_seed(1)
g  = generate("decoding ", 300, sample_greedy)
tp = generate("decoding ", 300, lambda z: sample_top_p(z, p=0.9, T=1.0))
print("== greedy ==\n", g[:300], sep="")
print(f"\n[greedy]  某 20-gram 最多重复 {longest_loop(g)} 次  <- 典型循环退化")
print("\n== top-p 0.9 ==\n", tp[:300], sep="")
print(f"\n[top-p]   某 20-gram 最多重复 {longest_loop(tp)} 次")

# ---- repetition penalty：HF 约定（正 logit 除以 r，负 logit 乘以 r）----
def greedy_with_rep_penalty(logits, recent_ids, r=1.5):
    z = logits.clone()
    for i in set(recent_ids):
        z[i] = z[i] / r if z[i] > 0 else z[i] * r
    return int(torch.argmax(z).item())

@torch.no_grad()
def generate_rp(prompt, n, r):
    ids = [stoi[c] for c in prompt]
    for _ in range(n):
        ctx = torch.tensor(ids[-BLOCK:]).unsqueeze(0)
        logits = model(ctx)[0, -1]
        ids.append(greedy_with_rep_penalty(logits, ids[-12:], r))  # 只罚最近 12 个
    return "".join(itos[i] for i in ids)

rp = generate_rp("decoding ", 300, r=1.5)
print("\n== greedy + repetition penalty r=1.5 (window=12) ==\n", rp[:300], sep="")
print(f"\n[rep-pen] 某 20-gram 最多重复 {longest_loop(rp)} 次",
      " <- 循环被打破，但注意拼写/用词开始变形：这就是 heuristic 惩罚的副作用",
      "\n(字符级惩罚会误伤高频字母——词级模型上副作用同理，只是更隐蔽)")

## Speculative decoding 模拟 [Leviathan 2022]

draft 模型 $q$ 起草、target 模型 $p$ 验证。单步**修正版接受-拒绝采样**：

1. 采 $x \sim q$；以概率 $\min\!\big(1, \frac{p(x)}{q(x)}\big)$ 接受；
2. 若拒绝，从残差分布 $p'(x) = \mathrm{norm}\big(\max(0,\, p(x) - q(x))\big)$ 重采。

**定理（无损性）**：输出严格服从 $p$，因为 $q(x)\min(1, \frac{p(x)}{q(x)}) + (1-\beta)\,p'(x) = \min(p,q) + \max(0, p-q) = p(x)$，其中接受率 $\beta = \sum_x \min(p(x), q(x)) = 1 - \mathrm{TV}(p, q)$。

下面用两个固定 categorical 分布（不需要真的跑两个模型——数学只关心分布本身），蒙特卡洛 10000 次验证：① 输出分布与 $p$ 的 TV 距离 ≈ 0；② 经验接受率 ≈ 理论值 $\beta$。

In [ ]:
torch.manual_seed(42)

p_target = torch.tensor([0.30, 0.20, 0.15, 0.10, 0.10, 0.05, 0.05, 0.05])  # 大模型
q_draft  = torch.tensor([0.20, 0.25, 0.10, 0.15, 0.05, 0.10, 0.05, 0.10])  # 小模型
assert abs(p_target.sum() - 1) < 1e-6 and abs(q_draft.sum() - 1) < 1e-6

beta_theory = torch.minimum(p_target, q_draft).sum().item()   # 理论接受率
residual = torch.clamp(p_target - q_draft, min=0.0)
residual = residual / residual.sum()                          # p'(x)
print(f"理论接受率 beta = sum min(p,q) = {beta_theory:.3f}  "
      f"(= 1 - TV(p,q) = {1 - 0.5 * (p_target - q_draft).abs().sum().item():.3f})")

def speculative_step():
    # 返回 (输出 token, 是否接受了草稿)
    x = int(torch.multinomial(q_draft, 1).item())
    if torch.rand(()) < min(1.0, p_target[x].item() / q_draft[x].item()):
        return x, True
    return int(torch.multinomial(residual, 1).item()), False

N = 10000
counts, accepts = torch.zeros(8), 0
for _ in range(N):
    x, ok = speculative_step()
    counts[x] += 1
    accepts += ok
emp = counts / N
tv = 0.5 * (emp - p_target).abs().sum().item()
print(f"经验输出分布 : {[f'{x:.3f}' for x in emp.tolist()]}")
print(f"target p     : {[f'{x:.3f}' for x in p_target.tolist()]}")
print(f"TV(经验, p)  = {tv:.4f}   经验接受率 = {accepts / N:.3f} (理论 {beta_theory:.3f})")
assert tv < 0.02, "speculative 的输出分布必须与 p 一致（无损性定理）"
assert abs(accepts / N - beta_theory) < 0.02
print("✅ 验证通过：输出分布 = p（与 draft 模型 q 无关），接受率 = 1 - TV(p, q)")

## ✏️ 练习 1：实现 `top_p_filter(logits, p)`

实现 nucleus 截断的**过滤器**（不采样）：返回与 `logits` 同形状的张量，nucleus 内的 token 保留原 logit，其余置 `-inf`。

**约定**（与 [Holtzman 2019] 一致）：按概率降序累积，保留"累计概率**首次达到 ≥ p** 的最小集合"——即**恰好跨过 p 阈值的那个 token 必须保留**。

提示：`torch.sort(descending=True)` → `softmax` → `cumsum`；"exclusive cumsum"（`cumsum - probs`，即每个 token *之前*已累计的质量）`< p` 就是保留条件，可保证首位 token 永远保留。10 行以内可完成。

In [ ]:
def top_p_filter(logits, p):
    # 返回过滤后的 logits：nucleus 内保留原值，nucleus 外置 -inf
    # TODO: 1) 对 logits 降序排序  2) softmax 得概率、cumsum 得累计
    #       3) exclusive cumsum < p 得到保留掩码  4) 映射回原始顺序
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测：固定分布 [0.5, 0.25, 0.125, 0.0625, 0.0625]（乱序摆放）----
probs5 = torch.tensor([0.0625, 0.5, 0.125, 0.0625, 0.25])   # 降序应为 idx 1,4,2,{0,3}
lg5 = torch.log(probs5)

def kept_set(out):
    return set(torch.where(torch.isfinite(out))[0].tolist())

assert kept_set(top_p_filter(lg5, 0.4))    == {1},           "p=0.4: 只留 0.5"
assert kept_set(top_p_filter(lg5, 0.7499)) == {1, 4},        "0.5+0.25=0.75 已跨过 0.7499"
assert kept_set(top_p_filter(lg5, 0.7501)) == {1, 4, 2},     "0.75<0.7501, 还需 0.125 跨过阈值"
assert kept_set(top_p_filter(lg5, 1.0))    == {0, 1, 2, 3, 4}, "p=1.0: 全保留"
out = top_p_filter(lg5, 0.7501)
assert all(out[i].item() == lg5[i].item() for i in [1, 4, 2]), "保留的 logit 必须原值不变"
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `apply_repetition_penalty(logits, prev_ids, penalty)`

按 HF `transformers` 约定实现 repetition penalty：对每个出现在 `prev_ids` 中的 token id，

- 若该 token 的 logit **> 0**：除以 `penalty`；
- 否则（**≤ 0**）：乘以 `penalty`。

两个分支都让该 token 更不可能被采到。要求：返回**新张量**（不要原地修改输入）；同一 id 在 `prev_ids` 中重复出现只惩罚一次（提示：先 `set(prev_ids)` 去重）。10 行以内可完成。

In [ ]:
def apply_repetition_penalty(logits, prev_ids, penalty):
    # TODO: clone logits；对 set(prev_ids) 中每个 id 按符号分支施加惩罚；返回新张量
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
lg = torch.tensor([2.0, -1.0, 0.0, 4.0])
out = apply_repetition_penalty(lg, [0, 1, 1, 2], penalty=2.0)
assert torch.allclose(out, torch.tensor([1.0, -2.0, 0.0, 4.0])), \
    "正 logit 除以 r、负 logit 乘以 r、0 不变（0*r=0）、未出现的 id 不动"
assert torch.allclose(lg, torch.tensor([2.0, -1.0, 0.0, 4.0])), "不得原地修改输入"
assert torch.allclose(apply_repetition_penalty(lg, [0, 0, 0], 2.0),
                      torch.tensor([1.0, -1.0, 0.0, 4.0])), "重复 id 只惩罚一次"
assert torch.allclose(apply_repetition_penalty(lg, [0, 1, 2, 3], 1.0), lg), \
    "penalty=1.0 应为恒等变换"
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `expected_speedup(accept_rate, gamma)`

[Leviathan 2022] 的期望产出公式：每 token 接受率为 $\alpha$（近似独立）、草稿长度为 $\gamma$ 时，一次"起草 + target 验证"迭代的期望产出 token 数（= 相对于逐 token 解码、忽略 draft 开销时的加速比）：

$$\mathbb{E}[\#\text{tokens}] = 1 + \alpha + \alpha^2 + \cdots + \alpha^{\gamma} = \frac{1 - \alpha^{\gamma+1}}{1 - \alpha}$$

实现它（输入 `accept_rate` $\in [0,1)$、整数 `gamma` ≥ 0，返回 float）。直觉自查：$\alpha = 0$ 时无论 $\gamma$ 多大都只值 1（草稿全废，只剩兜底重采的 1 个）。

In [ ]:
def expected_speedup(accept_rate, gamma):
    # TODO: 实现 (1 - accept_rate ** (gamma + 1)) / (1 - accept_rate)
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
assert abs(expected_speedup(0.0, 4) - 1.0) < 1e-9,  "alpha=0: 草稿全拒, 只产 1 个"
assert abs(expected_speedup(0.5, 1) - 1.5) < 1e-9,  "1 + 0.5"
assert abs(expected_speedup(0.8, 4) - 3.3616) < 1e-4, "(1-0.8^5)/0.2"
assert abs(expected_speedup(0.9, 0) - 1.0) < 1e-9,  "gamma=0: 没有草稿可验证"
assert expected_speedup(0.9, 8) > expected_speedup(0.5, 8), "alpha 越高加速越大"
print("✅ 练习 3 通过")

## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def top_p_filter(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    excl = torch.cumsum(probs, dim=-1) - probs   # 每个 token 之前的累计质量
    keep_sorted = excl < p                       # 跨过阈值的 token 被保留
    out = torch.full_like(logits, float("-inf"))
    kept_idx = sorted_idx[keep_sorted]
    out[kept_idx] = logits[kept_idx]
    return out

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def apply_repetition_penalty(logits, prev_ids, penalty):
    out = logits.clone()
    for i in set(int(t) for t in prev_ids):
        out[i] = out[i] / penalty if out[i] > 0 else out[i] * penalty
    return out

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def expected_speedup(accept_rate, gamma):
    # 等比数列求和 1 + a + ... + a^gamma；a=0 时也成立 (=1)
    return (1.0 - accept_rate ** (gamma + 1)) / (1.0 - accept_rate)

## 小结

- **解码策略 ≠ 模型**：同一组 logits，greedy / temperature / top-k / top-p 给出截然不同的文本；评测时解码参数是配置的一部分，必须报告并对齐。
- **likelihood trap** [Holtzman 2019]：最大似然序列是退化的重复文本，greedy/beam 在开放生成中失效；**top-p** 按概率质量自适应截断，比固定 **top-k** [Fan 2018] 更稳。
- **repetition / frequency / presence penalty** 是无分布保证的 heuristic——你已亲眼看到它打破循环的同时损伤文本质量。
- **speculative decoding** [Leviathan 2022] 是无损加速：你已用蒙特卡洛验证输出分布严格等于 target 分布 $p$，接受率 $= 1 - \mathrm{TV}(p,q)$，期望产出 $\frac{1-\alpha^{\gamma+1}}{1-\alpha}$。

**下一步 → 模块 05 · Scaling Laws：亲手拟合**。解码解决"训练好的模型怎么用"；下一模块回答"算力预算给定时，模型该训多大、数据该喂多少"——亲手拟合 Kaplan/Chinchilla 幂律。

---
## 🎯 真实数据胶囊题：真实下一字符分布上的 top-p (nucleus) 采样

用真实 tiny-shakespeare 统计出一个真实的“下一字符概率分布”，实现 nucleus (top-p) 过滤——保留累计概率达 p 的最小集合，其余置零重归一。这是现代 LLM 解码的默认。

> 本题为本模块新增的**真实数据**练习：自包含，直接用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, urllib.request, numpy as np
CACHE=os.path.expanduser("~/.llm_internals_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def shakespeare():
    return open(_fetch("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

txt=shakespeare(); chars=sorted(set(txt)); V=len(chars); stoi={c:i for i,c in enumerate(chars)}
ids=np.array([stoi[c] for c in txt])
# 真实分布：字符 ' ' 之后下一个字符的经验分布
prev=stoi[' ']; nxt=ids[1:][ids[:-1]==prev]
p_real=np.bincount(nxt, minlength=V).astype(float); p_real/=p_real.sum()
print(f"真实'空格后'下一字符分布 top3:", sorted(zip(p_real, chars))[-3:])

**练习**：实现 `top_p_filter(probs, p)`：保留按概率降序累计达到 `p` 的最小 token 集合，其余置 0，再重新归一化。返回新分布。

In [ ]:
def top_p_filter(probs, p):
    # TODO: 降序排, 累计>=p 处截断, 其余置0, 重归一
    raise NotImplementedError


In [ ]:
# 自测
q = top_p_filter(p_real, 0.9)
assert abs(q.sum()-1.0) < 1e-9, "应重新归一"
assert (q>0).sum() <= (p_real>0).sum(), "保留的不多于原非零数"
assert (q>0).sum() < V, "应过滤掉长尾"
# top-p=1.0 应几乎不变（保留全部）
assert np.allclose(top_p_filter(p_real,1.0), p_real, atol=1e-9)
# 最高概率 token 一定保留
assert q[p_real.argmax()]>0
print(f"top-p=0.9 把候选从 {(p_real>0).sum()} 个截到 {(q>0).sum()} 个 ✓")


### 📖 参考答案

In [ ]:
def top_p_filter(probs, p):
    idx=np.argsort(-probs); cum=np.cumsum(probs[idx])
    k=np.searchsorted(cum, p)+1
    keep=idx[:k]; q=np.zeros_like(probs); q[keep]=probs[keep]
    return q/q.sum()
print("✓ nucleus 采样：动态截断长尾，比 top-k 更自适应")